# Gradient Boosting (GBM)

In this notebook, we will implement **Gradient Boosting** for Regression. First, we will build a simplified mathematical version from scratch using basic `DecisionTreeRegressor`s to predict residuals, proving the underlying intuition. Then, we will use the production-level `sklearn.ensemble.GradientBoostingRegressor` to solve a non-linear problem.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_squared_error

# Set random seed for reproducibility
np.random.seed(42)

### 1. Generating a Non-Linear Dataset
Let's create a synthetic dataset that has a clear non-linear pattern (a quadratic function with some noise).

In [ ]:
X = np.random.rand(100, 1) - 0.5
y = 3 * X[:, 0]**2 + 0.05 * np.random.randn(100)

plt.scatter(X, y, color='blue', alpha=0.6, edgecolors='k')
plt.title('Synthetic Non-Linear Data')
plt.xlabel('X')
plt.ylabel('y')
plt.show()

### 2. Gradient Boosting From Scratch (Intuition)

The core idea is to train sequentially. 
- **Step 1:** Train a model on the data. Get predictions.
- **Step 2:** Calculate the residuals (Actual - Predicted).
- **Step 3:** Train a *new* model to predict the *residuals*.
- **Step 4:** Add the new model's predictions to the overall prediction.
- Repeat.

In [ ]:
# 1. Initial Model (Base Model)
tree_reg1 = DecisionTreeRegressor(max_depth=2, random_state=42)
tree_reg1.fit(X, y)

# Calculate residuals
y2 = y - tree_reg1.predict(X)

# 2. Second Model (Trains on Residuals)
tree_reg2 = DecisionTreeRegressor(max_depth=2, random_state=42)
tree_reg2.fit(X, y2)

# Calculate new residuals
y3 = y2 - tree_reg2.predict(X)

# 3. Third Model (Trains on New Residuals)
tree_reg3 = DecisionTreeRegressor(max_depth=2, random_state=42)
tree_reg3.fit(X, y3)

print("Finished training 3 sequential trees on residuals.")

Now, to make a prediction on a new data point, we simply sum the predictions from all three trees:

In [ ]:
# Let's visualize the final prediction
X_new = np.linspace(-0.5, 0.5, 500).reshape(-1, 1)

y_pred = sum(tree.predict(X_new) for tree in (tree_reg1, tree_reg2, tree_reg3))

plt.scatter(X, y, color='blue', alpha=0.6, edgecolors='k', label='Actual Data')
plt.plot(X_new, y_pred, color='red', linewidth=2, label='Ensemble Prediction')
plt.title('Prediction of 3-Tree Gradient Boosting Ensemble')
plt.legend()
plt.show()

### 3. Gradient Boosting using Scikit-Learn

Instead of building it manually, we can use Scikit-Learn's highly optimized `GradientBoostingRegressor`.

In [ ]:
# Using Scikit-Learn's implementation
gbrt = GradientBoostingRegressor(max_depth=2, n_estimators=3, learning_rate=1.0, random_state=42)
gbrt.fit(X, y)

y_pred_sklearn = gbrt.predict(X_new)

plt.scatter(X, y, color='blue', alpha=0.6, edgecolors='k', label='Actual Data')
plt.plot(X_new, y_pred_sklearn, color='green', linewidth=2, label='Sklearn GBM Prediction')
plt.title('Sklearn Gradient Boosting (learning_rate=1.0, n_estimators=3)')
plt.legend()
plt.show()

### 4. The Effect of Learning Rate (Shrinkage)
A lower learning rate means each tree contributes less to the final prediction, requiring more trees (`n_estimators`) in the ensemble to fit the training set, but usually leading to better generalization.

In [ ]:
gbrt_slow = GradientBoostingRegressor(max_depth=2, n_estimators=200, learning_rate=0.1, random_state=42)
gbrt_slow.fit(X, y)

y_pred_slow = gbrt_slow.predict(X_new)

plt.scatter(X, y, color='blue', alpha=0.6, edgecolors='k')
plt.plot(X_new, y_pred_slow, color='purple', linewidth=2, label='learning_rate=0.1, n_estimators=200')
plt.title('Gradient Boosting with Shrinkage')
plt.legend()
plt.show()